# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 11.1 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64, pickle
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference

In [5]:
torch.set_num_threads(1)

In [6]:
TASK_ID = "task361"
CH = 10
H = W = 30
LOCAL_TASK_JSON = Path("/mnt/data/task361(2).json")
LOCAL_ALT_TASK_JSON = Path("/mnt/data/task361.json")
KAGGLE_TASK_JSON = Path(COMPETITION) / f"{TASK_ID}.json"
TASK_JSON = LOCAL_TASK_JSON if LOCAL_TASK_JSON.exists() else (LOCAL_ALT_TASK_JSON if LOCAL_ALT_TASK_JSON.exists() else KAGGLE_TASK_JSON)

OUT_DIR = Path.cwd() / "task361_core_square_c4_active_canvas_30x30"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ONNX_PATH = OUT_DIR / f"{TASK_ID}.onnx"
SUBMISSION_PATH = Path.cwd() / "submission.zip"
SUMMARY_PATH = OUT_DIR / f"{TASK_ID}_core_square_c4_validation_summary.json"
RULE_DESCRIPTION = "maximal filled 2x2/3x3 core-square detection + C4 rotational completion + active-canvas masking"

with TASK_JSON.open('r') as f:
    task = json.load(f)

print('task json:', TASK_JSON)
print('train/test/arc-gen:', len(task['train']), len(task['test']), len(task.get('arc-gen', [])))

task json: /kaggle/input/competitions/neurogolf-2026/task361.json
train/test/arc-gen: 3 1 262


In [7]:
def grid_to_tensor_zero_padded(grid, h=H, w=W, ch=CH):
    """Convert an ARC grid to [1,10,30,30].
    Inside the real grid: one-hot, including background color 0.
    Outside the real grid: all-zero across every channel.
    """
    x = np.zeros((1, ch, h, w), dtype=np.float32)
    gh, gw = len(grid), len(grid[0])
    assert gh <= h and gw <= w
    for r, row in enumerate(grid):
        for c, v in enumerate(row):
            x[0, int(v), r, c] = 1.0
    return x

def tensor_to_grid(tensor, h, w):
    return tensor[0, :, :h, :w].argmax(axis=0).astype(np.int64).tolist()

def vi_shape(vi):
    dims = []
    for d in vi.type.tensor_type.shape.dim:
        if d.HasField('dim_value'):
            dims.append(int(d.dim_value))
        elif d.dim_param:
            dims.append(str(d.dim_param))
        else:
            dims.append(None)
    return dims

In [8]:
class Task361CoreSquareC4Model(nn.Module):
    """Static symbolic model for task361.

    The model avoids visible-template lookup. It detects the central dense square
    core and performs a C4 rotational closure of every non-background pixel.

    Full-core detection:
      - 3x3 all-nonzero square, if present: integer center.
      - otherwise 2x2 all-nonzero square: half-integer center.

    The C4 orbit is implemented by four static GridSample calls over the full
    [1,10,30,30] tensor. Padding is handled by active_canvas = sum(input_channels)>0.
    """
    def __init__(self, h=H, w=W):
        super().__init__()
        self.h = h
        self.w = w

        R = torch.arange(h, dtype=torch.float32).view(1, h, 1).expand(1, h, w)
        C = torch.arange(w, dtype=torch.float32).view(1, 1, w).expand(1, h, w)
        self.register_buffer('R2', 2.0 * R)
        self.register_buffer('C2', 2.0 * C)
        self.register_buffer('k3', torch.ones((1, 1, 3, 3), dtype=torch.float32))
        self.register_buffer('k2', torch.ones((1, 1, 2, 2), dtype=torch.float32))

        # 3x3 top-left positions are 28x28. Their centers are (r+1, c+1),
        # represented in doubled coordinates as (2r+2, 2c+2).
        r28 = torch.arange(h - 2, dtype=torch.float32).view(h - 2, 1).expand(h - 2, w - 2).reshape(-1) * 2.0 + 2.0
        c28 = torch.arange(w - 2, dtype=torch.float32).view(1, w - 2).expand(h - 2, w - 2).reshape(-1) * 2.0 + 2.0
        self.register_buffer('r2_28', r28)
        self.register_buffer('c2_28', c28)

        # 2x2 top-left positions are 29x29. Their centers are (r+0.5, c+0.5),
        # represented in doubled coordinates as (2r+1, 2c+1).
        r29 = torch.arange(h - 1, dtype=torch.float32).view(h - 1, 1).expand(h - 1, w - 1).reshape(-1) * 2.0 + 1.0
        c29 = torch.arange(w - 1, dtype=torch.float32).view(1, w - 1).expand(h - 1, w - 1).reshape(-1) * 2.0 + 1.0
        self.register_buffer('r2_29', r29)
        self.register_buffer('c2_29', c29)

    def _grid(self, sr2, sc2):
        # Convert doubled source row/col coordinates into GridSample's normalized x/y grid.
        sr = sr2 * 0.5
        sc = sc2 * 0.5
        gx = sc * (2.0 / float(self.w - 1)) - 1.0
        gy = sr * (2.0 / float(self.h - 1)) - 1.0
        return torch.stack([gx, gy], dim=-1)

    def forward(self, x):
        active = (x.sum(dim=1, keepdim=True) > 0.5).float()
        nz = (x[:, 1:, :, :].sum(dim=1, keepdim=True) > 0.5).float()

        full3 = (F.conv2d(nz, self.k3) > 8.5).float()
        flat3 = full3.reshape(1, 784)
        max3 = flat3.max(dim=1, keepdim=True).values
        idx3 = torch.argmax(flat3, dim=1)
        cr3 = self.r2_28[idx3].reshape(1, 1, 1)
        cc3 = self.c2_28[idx3].reshape(1, 1, 1)

        full2 = (F.conv2d(nz, self.k2) > 3.5).float()
        flat2 = full2.reshape(1, 841)
        idx2 = torch.argmax(flat2, dim=1)
        cr2h = self.r2_29[idx2].reshape(1, 1, 1)
        cc2h = self.c2_29[idx2].reshape(1, 1, 1)

        has3 = (max3 > 0.5).float().reshape(1, 1, 1)
        cr2 = has3 * cr3 + (1.0 - has3) * cr2h
        cc2 = has3 * cc3 + (1.0 - has3) * cc2h

        R2 = self.R2
        C2 = self.C2
        nzch = x[:, 1:, :, :]

        # For every output position, sample the source under the four inverse C4 rotations.
        sr0, sc0 = R2, C2
        sr1 = cr2 + (C2 - cc2)
        sc1 = cc2 - (R2 - cr2)
        sr2 = 2.0 * cr2 - R2
        sc2 = 2.0 * cc2 - C2
        sr3 = cr2 - (C2 - cc2)
        sc3 = cc2 + (R2 - cr2)

        y0 = F.grid_sample(nzch, self._grid(sr0, sc0), mode='nearest', padding_mode='zeros', align_corners=True)
        y1 = F.grid_sample(nzch, self._grid(sr1, sc1), mode='nearest', padding_mode='zeros', align_corners=True)
        y2 = F.grid_sample(nzch, self._grid(sr2, sc2), mode='nearest', padding_mode='zeros', align_corners=True)
        y3 = F.grid_sample(nzch, self._grid(sr3, sc3), mode='nearest', padding_mode='zeros', align_corners=True)

        out_nz = torch.clamp(y0 + y1 + y2 + y3, 0.0, 1.0)
        any_nz = (out_nz.sum(dim=1, keepdim=True) > 0.5).float()
        ch0 = active * (1.0 - any_nz)
        return torch.cat([ch0, out_nz], dim=1) * active

model = Task361CoreSquareC4Model().eval()
print('model ready:', RULE_DESCRIPTION)

model ready: maximal filled 2x2/3x3 core-square detection + C4 rotational completion + active-canvas masking


In [9]:
# Quick PyTorch-side sanity check before ONNX export.
for split in ['train', 'test', 'arc-gen']:
    ok = 0
    for ex in task[split]:
        x = torch.from_numpy(grid_to_tensor_zero_padded(ex['input']))
        y = model(x).detach().numpy()
        exp = grid_to_tensor_zero_padded(ex['output'])
        ok += int(np.array_equal((y > 0.5).astype(np.float32), exp))
    print('torch', split, ok, '/', len(task[split]))

torch train 3 / 3
torch test 1 / 1
torch arc-gen 262 / 262


In [10]:
dummy = torch.from_numpy(grid_to_tensor_zero_padded(task['test'][0]['input']))

torch.onnx.export(
    model,
    dummy,
    str(ONNX_PATH),
    input_names=['input'],
    output_names=['output'],
    opset_version=17,
    do_constant_folding=True,
    dynamic_axes=None,
    dynamo=False,
)

onnx_model = onnx.load(str(ONNX_PATH))
onnx_model = onnx.shape_inference.infer_shapes(onnx_model)
for i, dim_value in enumerate([1, 10, 30, 30]):
    dim = onnx_model.graph.output[0].type.tensor_type.shape.dim[i]
    dim.ClearField('dim_param')
    dim.dim_value = dim_value
onnx.save(onnx_model, str(ONNX_PATH))
onnx.checker.check_model(str(ONNX_PATH))

print('ONNX:', ONNX_PATH)
print('size bytes:', ONNX_PATH.stat().st_size)

/tmp/ipykernel_16/1921527393.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX: /kaggle/working/task361_core_square_c4_active_canvas_30x30/task361.onnx
size bytes: 44108


In [11]:
onnx_model = onnx.load(str(ONNX_PATH))
ops = collections.Counter(node.op_type for node in onnx_model.graph.node)
forbidden = {'Loop', 'Scan', 'NonZero', 'Unique', 'Script', 'Function'}
empty_inputs = [
    (node.name, node.op_type, list(node.input))
    for node in onnx_model.graph.node
    if any(inp == '' for inp in node.input)
]
bad_shapes = []
for vi in list(onnx_model.graph.input) + list(onnx_model.graph.value_info) + list(onnx_model.graph.output):
    shp = vi_shape(vi)
    if any(d is None or isinstance(d, str) for d in shp):
        bad_shapes.append((vi.name, shp))

print('input shape:', vi_shape(onnx_model.graph.input[0]))
print('output shape:', vi_shape(onnx_model.graph.output[0]))
print('ONNX size:', ONNX_PATH.stat().st_size)
print('ops:', dict(ops))
print('forbidden ops:', sorted(forbidden & set(ops)))
print('empty optional inputs:', len(empty_inputs))
print('non-static tensor shapes:', len(bad_shapes))

assert vi_shape(onnx_model.graph.input[0]) == [1, 10, 30, 30]
assert vi_shape(onnx_model.graph.output[0]) == [1, 10, 30, 30]
assert not (forbidden & set(ops))
assert not empty_inputs
assert not bad_shapes
assert ONNX_PATH.stat().st_size < 1_400_000

input shape: [1, 10, 30, 30]
output shape: [1, 10, 30, 30]
ONNX size: 44108
ops: {'Constant': 48, 'ReduceSum': 3, 'Greater': 6, 'Cast': 6, 'Slice': 1, 'Conv': 2, 'Reshape': 7, 'ReduceMax': 1, 'ArgMax': 2, 'Gather': 4, 'Mul': 20, 'Sub': 14, 'Add': 7, 'GridSample': 4, 'Unsqueeze': 6, 'Concat': 4, 'Clip': 1}
forbidden ops: []
empty optional inputs: 0
non-static tensor shapes: 0


In [12]:
sess_options = ort.SessionOptions()
sess_options.intra_op_num_threads = 1
sess_options.inter_op_num_threads = 1
sess = ort.InferenceSession(str(ONNX_PATH), sess_options=sess_options, providers=['CPUExecutionProvider'])

def validate_examples(examples):
    tensor_ok = 0
    grid_ok = 0
    outside_input_active_zero_ok = 0
    outside_expected_canvas_zero_ok = 0
    bad = []
    for i, ex in enumerate(examples):
        x = grid_to_tensor_zero_padded(ex['input'])
        y = sess.run(None, {'input': x})[0]
        exp = grid_to_tensor_zero_padded(ex['output'])
        pred_bin = (y > 0.5).astype(np.float32)

        if np.array_equal(pred_bin, exp):
            tensor_ok += 1
        else:
            bad.append(i)

        h, w = len(ex['output']), len(ex['output'][0])
        pred_grid = tensor_to_grid(pred_bin, h, w)
        if pred_grid == ex['output']:
            grid_ok += 1

        input_active = x.sum(axis=1, keepdims=True) > 0.5
        expected_active = exp.sum(axis=1, keepdims=True) > 0.5
        if np.all(np.abs(y * (~input_active)) < 1e-5):
            outside_input_active_zero_ok += 1
        if np.all(np.abs(y * (~expected_active)) < 1e-5):
            outside_expected_canvas_zero_ok += 1

    return {
        'tensor_exact_zero_padded': [tensor_ok, len(examples)],
        'grid_argmax_inside_output_canvas': [grid_ok, len(examples)],
        'outside_input_active_all_channels_zero': [outside_input_active_zero_ok, len(examples)],
        'outside_expected_output_canvas_all_channels_zero': [outside_expected_canvas_zero_ok, len(examples)],
        'bad_indices': bad[:20],
    }

def validate_split(split):
    return validate_examples(task[split])

rng = random.Random(0)
arcgen_indices = list(range(len(task['arc-gen'])))
rng.shuffle(arcgen_indices)
holdout_n = max(1, int(math.ceil(0.60 * len(arcgen_indices))))
arcgen_holdout = [task['arc-gen'][i] for i in arcgen_indices[:holdout_n]]

summary = {
    'task_id': TASK_ID,
    'rule': RULE_DESCRIPTION,
    'onnx_path': str(ONNX_PATH),
    'onnx_size_bytes': ONNX_PATH.stat().st_size,
    'input_shape': vi_shape(onnx_model.graph.input[0]),
    'output_shape': vi_shape(onnx_model.graph.output[0]),
    'ops': dict(ops),
    'forbidden_ops': sorted(forbidden & set(ops)),
    'empty_optional_inputs': len(empty_inputs),
    'non_static_tensor_shapes': len(bad_shapes),
    'arc_gen_holdout_policy': 'deterministic random seed 0, 60% of arc-gen; full arc-gen also validated',
    'validation': {
        'train': validate_split('train'),
        'test': validate_split('test'),
        'arc-gen_60pct_holdout': validate_examples(arcgen_holdout),
        'arc-gen_full': validate_split('arc-gen'),
    },
}

print(json.dumps(summary, indent=2)[:8000])
with SUMMARY_PATH.open('w') as f:
    json.dump(summary, f, indent=2)

for split_name, result in summary['validation'].items():
    assert result['tensor_exact_zero_padded'][0] == result['tensor_exact_zero_padded'][1], split_name
    assert result['grid_argmax_inside_output_canvas'][0] == result['grid_argmax_inside_output_canvas'][1], split_name
    assert result['outside_input_active_all_channels_zero'][0] == result['outside_input_active_all_channels_zero'][1], split_name
    assert result['outside_expected_output_canvas_all_channels_zero'][0] == result['outside_expected_output_canvas_all_channels_zero'][1], split_name

{
  "task_id": "task361",
  "rule": "maximal filled 2x2/3x3 core-square detection + C4 rotational completion + active-canvas masking",
  "onnx_path": "/kaggle/working/task361_core_square_c4_active_canvas_30x30/task361.onnx",
  "onnx_size_bytes": 44108,
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "ops": {
    "Constant": 48,
    "ReduceSum": 3,
    "Greater": 6,
    "Cast": 6,
    "Slice": 1,
    "Conv": 2,
    "Reshape": 7,
    "ReduceMax": 1,
    "ArgMax": 2,
    "Gather": 4,
    "Mul": 20,
    "Sub": 14,
    "Add": 7,
    "GridSample": 4,
    "Unsqueeze": 6,
    "Concat": 4,
    "Clip": 1
  },
  "forbidden_ops": [],
  "empty_optional_inputs": 0,
  "non_static_tensor_shapes": 0,
  "arc_gen_holdout_policy": "deterministic random seed 0, 60% of arc-gen; full arc-gen also validated",
  "validation": {
    "train": {
      "tensor_exact_zero_padded": [
        3,
        3
      ],
      "grid_argmax_inside_output_canvas":

In [13]:
with zipfile.ZipFile(SUBMISSION_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(ONNX_PATH, arcname=f'{TASK_ID}.onnx')

print('Wrote:', SUBMISSION_PATH)
print('Zip contents:', zipfile.ZipFile(SUBMISSION_PATH).namelist())
assert zipfile.ZipFile(SUBMISSION_PATH).namelist() == [f'{TASK_ID}.onnx']

Wrote: /kaggle/working/submission.zip
Zip contents: ['task361.onnx']
